In [79]:
import numpy as np
import pandas as pd

## products

In [80]:
import os
print(os.getcwd())

d:\TAI LIEU\HK6\BI\ĐỒ ÁN CUỐI KÌ\BI-2026\data_database


In [81]:
df1 = pd.read_csv("raw/dim_products.csv")
df1.head()

,product_name,product_id,category
0,AM Milk 500,25891101,Dairy
1,AM Milk 250,25891102,Dairy
2,AM Milk 100,25891103,Dairy
3,AM Butter 100,25891201,Dairy
4,AM Butter 250,25891202,Dairy


In [82]:
## convert datatype

df1['product_id'] = df1['product_id'].astype(str)
df1['product_name'] = df1['product_name'].astype(str)
df1['category'] = df1['category'].astype(str)

In [83]:
df1.isnull().sum()

product_name    0
product_id      0
category        0
dtype: int64

In [84]:
df1 = df1.drop_duplicates()

### Thêm cột price

In [85]:
np.random.seed(42)

## Xác định price_range
def get_price_range(category):
    if category == 'Food':
        return (20000, 100000)
    elif category == 'Beverages':
        return (10000, 50000)
    elif category == 'Dairy':
        return (50000, 150000)
    elif category == 'Household':
        return (100000, 500000)
    else:
        return (30000, 200000)

## tạo price_map
price_map = {}

for _, row in df1.iterrows():
    product_id = row['product_id']
    category = row['category']
    
    if product_id not in price_map:
        low, high = get_price_range(category)
        price = np.random.uniform(low, high)
        price_map[product_id] = int(round(price / 1000) * 1000)

df1['price'] = df1['product_id'].map(price_map)

### Thêm cột cost _ Chi phí tạo ra sản phẩm

In [86]:
## cost = price × (1 - margin) ; margin theo từng category

np.random.seed(42)
def get_margin_range(category):
    if category == 'Food':
        return (0.1, 0.25)
    elif category == 'Beverages':
        return (0.2, 0.4)
    elif category == 'Dairy':
        return (0.15, 0.3)
    elif category == 'Household':
        return (0.35, 0.6)
    else:
        return (0.2, 0.4)

def generate_cost(row):
    low, high = get_margin_range(row['category'])
    
    margin = np.random.uniform(low, high)
    base_cost = row['price'] * (1 - margin)
    noise = np.random.uniform(0.97, 1.03)
    cost = base_cost * noise
    cost = int(round(cost / 1000) * 1000)
    return cost

df1['cost'] = df1.apply(generate_cost, axis=1)

### Check logic

In [87]:
## Gía theo product_id
print(df1.groupby('product_id')['price'].nunique().value_counts())

df1 = df1.drop_duplicates(subset=['product_id'])

price
1    18
Name: count, dtype: int64


In [88]:
### LOGIC CỦA COST

In [89]:
## cost > price
print((df1['cost'] >= df1['price']).sum()) 

0


In [90]:
## Check margin có hợp lí không
margin = (df1['price'] - df1['cost']) / df1['price']
print(margin.describe())

count    18.000000
mean      0.213811
std       0.066126
min       0.135135
25%       0.184679
50%       0.196200
75%       0.244318
max       0.378151
dtype: float64


In [91]:
## Một product không được có nhiều cost
df1.groupby('product_id')['cost'].nunique().value_counts()

cost
1    18
Name: count, dtype: int64

In [92]:
df1.head(21)

,product_name,product_id,category,price,cost
0,AM Milk 500,25891101,Dairy,87000,71000
1,AM Milk 250,25891102,Dairy,145000,108000
2,AM Milk 100,25891103,Dairy,123000,100000
3,AM Butter 100,25891201,Dairy,110000,95000
4,AM Butter 250,25891202,Dairy,66000,51000
5,AM Butter 500,25891203,Dairy,66000,57000
6,AM Ghee 250,25891301,Dairy,56000,40000
7,AM Ghee 150,25891302,Dairy,137000,111000
8,AM Ghee 100,25891303,Dairy,110000,89000
9,AM Curd 250,25891401,Dairy,121000,94000


In [93]:
df1.to_csv('products.csv', index=False)

## fact_order_lines

In [94]:
df2 = pd.read_csv("raw/fact_order_lines.csv")
df2.head()

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,In Full,On Time,On Time In Full
0,FMR34203601,"Tuesday, March 1, 2022",789203,25891601,110,"Friday, March 4, 2022","Friday, March 4, 2022",110,1,1,1
1,FMR32320302,"Tuesday, March 1, 2022",789320,25891203,347,"Wednesday, March 2, 2022","Wednesday, March 2, 2022",347,1,1,1
2,FMR33320501,"Tuesday, March 1, 2022",789320,25891203,187,"Thursday, March 3, 2022","Thursday, March 3, 2022",150,0,1,0
3,FMR34220601,"Tuesday, March 1, 2022",789220,25891203,235,"Friday, March 4, 2022","Friday, March 4, 2022",235,1,1,1
4,FMR33703603,"Tuesday, March 1, 2022",789703,25891203,176,"Thursday, March 3, 2022","Thursday, March 3, 2022",176,1,1,1


In [95]:
df2.dtypes

order_id                object
order_placement_date    object
customer_id              int64
product_id               int64
order_qty                int64
agreed_delivery_date    object
actual_delivery_date    object
delivery_qty             int64
In Full                  int64
On Time                  int64
On Time In Full          int64
dtype: object

In [96]:
df2['order_placement_date'] = pd.to_datetime(df2['order_placement_date'])
df2['agreed_delivery_date'] = pd.to_datetime(df2['agreed_delivery_date'])
df2['actual_delivery_date'] = pd.to_datetime(df2['actual_delivery_date'])

In [97]:
df2['order_qty'] = df2['order_qty'].astype(int)
df2['delivery_qty'] = df2['delivery_qty'].astype(int)

df2['order_id'] = df2['order_id'].astype(str)
df2['customer_id'] = df2['customer_id'].astype(str)
df2['product_id'] = df2['product_id'].astype(str)

In [98]:
df2.isnull().sum()

order_id                0
order_placement_date    0
customer_id             0
product_id              0
order_qty               0
agreed_delivery_date    0
actual_delivery_date    0
delivery_qty            0
In Full                 0
On Time                 0
On Time In Full         0
dtype: int64

In [99]:
df2 = df2.drop_duplicates()

### Check logic

In [100]:
## delivery ≤ order
df2 = df2[df2['delivery_qty'] <= df2['order_qty']]

In [101]:
## ngày giao ≥ ngày đặt
df2 = df2[df2['actual_delivery_date'] >= df2['order_placement_date']]

In [102]:
## agreed ≥ placement
df2 = df2[df2['agreed_delivery_date'] >= df2['order_placement_date']]

In [103]:
## không null ở key
df2 = df2.dropna(subset=['order_id', 'customer_id', 'product_id'])

In [104]:
## order_qty > 0
df2 = df2[df2['order_qty'] > 0]

In [105]:
## delivery_qty ≥ 0
df2 = df2[df2['delivery_qty'] >= 0]

In [106]:
## 1 order_id chỉ được thuộc về 1 customer
df2.groupby('order_id')['customer_id'].nunique().value_counts()

customer_id
1    31729
Name: count, dtype: int64

In [107]:
invalid_orders = df2.groupby('order_id')['customer_id'].nunique()
invalid_orders = invalid_orders[invalid_orders > 1]

df2 = df2[~df2['order_id'].isin(invalid_orders.index)]

### Xóa cột suy diễn

In [108]:
df2 = df2.drop(columns=['In Full', 'On Time', 'On Time In Full'])

### Tách ra 2 bảng orders và order lines

In [109]:
## DO NẾU ĐỂ 1 BẢNG NHƯ fact_order_lines THÌ SẼ BỊ TRƯỜNG HỢP 1 order_placmement_date TRONG 1 ORDER (data redundancy )

orders = df2[['order_id', 'customer_id', 'order_placement_date']].drop_duplicates()
order_lines = df2.copy()

## thêm primary key cho order_lines
order_lines = order_lines.reset_index(drop=True)
order_lines['order_line_id'] = order_lines.index + 1

order_lines = order_lines[[
    'order_line_id',
    'order_id',
    'product_id',
    'order_qty',
    'delivery_qty',
    'agreed_delivery_date',
    'actual_delivery_date'
]]

In [110]:
# ## MERGE price VÀO order_lines ĐỂ DÙNG CHO TÍNH REVENUE K CẦN JOIN NHIỀU LẦN

# order_lines = order_lines.merge(
#     df1[['product_id', 'price']],
#     on='product_id',
#     how='left'
# )
# assert order_lines['price'].isnull().sum() == 0
# print("Missing price:", order_lines['price'].isnull().sum())
# missing_product = set(df2['product_id']) - set(df1['product_id'])
# print(len(missing_product))

In [111]:
orders.to_csv("orders.csv", index=False)
order_lines.to_csv("order_lines.csv", index=False)

In [118]:
df2.head()

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150
3,FMR34220601,2022-03-01,789220,25891203,235,2022-03-04,2022-03-04,235
4,FMR33703603,2022-03-01,789703,25891203,176,2022-03-03,2022-03-03,176


## fact_target_orders

In [112]:
df3= pd.read_csv("raw/dim_targets_orders.csv")
df3.head()

,customer_id,ontime_target%,infull_target%,otif_target%
0,789201,87,81,70
1,789202,85,81,69
2,789203,92,76,70
3,789301,89,78,69
4,789303,88,78,69


In [113]:
## rename cho đún chuẩn
df3 = df3.rename(columns={
    'ontime_target%': 'ontime_target',
    'infull_target%': 'infull_target',
    'otif_target%': 'otif_target'
})

In [114]:
## convert datatype
df3['customer_id'] = df3['customer_id'].astype(str)
df3['ontime_target'] = df3['ontime_target'].astype(float)
df3['infull_target'] = df3['infull_target'].astype(float)
df3['otif_target'] = df3['otif_target'].astype(float)

In [115]:
## Chuẩn hóa phần trăm
df3['ontime_target'] = df3['ontime_target'] / 100
df3['infull_target'] = df3['infull_target'] / 100
df3['otif_target'] = df3['otif_target'] / 100

### check logic

In [116]:
df3[(df3['ontime_target'] < 0) | (df3['ontime_target'] > 1)]

df3['customer_id'].duplicated().sum()
df3 = df3.drop_duplicates(subset=['customer_id'])

In [117]:
df3.to_csv("targets_orders.csv", index=False)